# 04b · Defenses against the style attack

**Stage 4c.** Notebook 03b showed that both models, and DistilBERT above all, rely on source markers and tone.
Here we retrain both models with two defenses and test them again on the same attack:

| Variant | Training data | Input at prediction time |
|---|---|---|
| `baseline` | original training set (the models from notebooks 01–02) | as is |
| `normalised` | training set with source markers stripped | source markers stripped |
| `augmented` | original training set + 1,000 style-flipped rewrites of 500 training articles | as is |
| `combined` | stripped training set + the rewrites | source markers stripped |

**Normalisation** is free: a regex removes agency tags, publisher suffixes, `[VIDEO]` tags, image credits and URLs.
**Augmentation** costs LLM rewrites: real training articles rewritten in a sensational tone and fake ones in a
neutral tone (`flip2` and `flip3` from the same prompt as the attack), keeping their labels.

Each variant is scored on the clean test set (4,500 articles) and on the 03b attack set (the articles whose rewrites
passed the NLI check), so we can see what each defense **recovers** under attack and what it **costs** on clean data.

**Runtime:** T4 GPU, about 20 minutes (three DistilBERT trainings of ~5 minutes each). If Colab disconnects,
run all again: finished variants are cached on Drive and skipped.

### ▶ Keep this tab awake (tap play once)

Phone browsers suspend background tabs, which can freeze the notebook's progress display or drop the
connection while long cells run. The player below loops a **near-silent** 5‑second sound. Tap ▶ once
after starting *Run all*; it keeps playing in your browser while the other cells run.
It does **not** extend Colab's own runtime limits, so keep the phone charging and Colab open.

In [ ]:
import io, wave, base64
import numpy as np
from IPython.display import HTML, display

_sr = 8000
_noise = (np.random.default_rng(0).standard_normal(_sr * 5) * 20).astype(np.int16)   # ~-64 dBFS: inaudible
_buf = io.BytesIO()
with wave.open(_buf, "wb") as _w:
    _w.setnchannels(1); _w.setsampwidth(2); _w.setframerate(_sr); _w.writeframes(_noise.tobytes())
display(HTML(f'<audio controls loop autoplay src="data:audio/wav;base64,{base64.b64encode(_buf.getvalue()).decode()}"></audio>'
             '<div style="font-size:12px;color:gray">Keep-alive player: tap ▶ if it is not already playing.</div>'))

## 0 · Configuration

In [ ]:
SEED = 42
VARIANTS = ["baseline", "normalised", "augmented", "combined"]
AUG_CONDITIONS = ["flip2", "flip3"]
N_AUG_CHUNKS = 20
# TF-IDF + LR (C grid widened: in notebook 01 the best C sat at the top of the grid)
C_GRID = [1, 3, 10, 30, 100, 300]
# DistilBERT, same settings as notebook 02
MODEL_NAME = "distilbert-base-uncased"
MAX_LEN = 256
BATCH_SIZE = 32
EVAL_BATCH_SIZE = 128
EPOCHS = 2
LR = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_FRAC = 0.06
ATTACK_CONDITIONS = ["original", "stripped", "same", "flip1", "flip2", "flip3"]
PROJECT_DIRNAME = "NLP_style_attack"
REPO_RAW = "https://raw.githubusercontent.com/AfthabEK/NLP/HEAD"

## 1 · Setup

In [ ]:
import os, sys, json, time, random, urllib.request
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR = f"/content/drive/MyDrive/{PROJECT_DIRNAME}"
else:
    BASE_DIR = os.environ.get("FAKESTYLE_BASE_DIR", os.path.abspath(f"./{PROJECT_DIRNAME}"))
DIRS = {k: os.path.join(BASE_DIR, k) for k in ["data", "results", "models", "figures"]}
for d in DIRS.values():
    os.makedirs(d, exist_ok=True)

CODE_DIR = os.environ.get("FAKESTYLE_CODE_DIR") or "/content/fakestyle_src"
if not os.environ.get("FAKESTYLE_CODE_DIR"):
    os.makedirs(os.path.join(CODE_DIR, "fakestyle"), exist_ok=True)
    open(os.path.join(CODE_DIR, "fakestyle", "__init__.py"), "w").close()
    urllib.request.urlretrieve(f"{REPO_RAW}/src/fakestyle/text.py", os.path.join(CODE_DIR, "fakestyle", "text.py"))
sys.path.insert(0, CODE_DIR)
from fakestyle.text import strip_source_markers

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
print("Device:", torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "CPU (switch to a T4 runtime!)")
print("Outputs ->", BASE_DIR)

## 2 · Data: splits, augmentation rewrites, attack set

In [ ]:
splits = {n: pd.read_parquet(os.path.join(DIRS["data"], f"{n}.parquet")) for n in ["train", "val", "test"]}

def stripped_copy(d):
    out = d.copy()
    pairs = [strip_source_markers(a, b) for a, b in zip(out["title"], out["text"])]
    out["title"], out["text"] = [p[0] for p in pairs], [p[1] for p in pairs]
    return out

splits_stripped = {n: stripped_copy(d) for n, d in splits.items()}

# Augmentation rewrites (notebook 04a sample, rewritten by Claude; downloaded from the project repo)
aug_sample = pd.read_parquet(os.path.join(DIRS["data"], "aug_sample.parquet"))
local_dir = os.environ.get("FAKESTYLE_AUG_DIR")
rows, missing = [], []
for i in range(N_AUG_CHUNKS):
    name = f"chunk_{i:02d}.jsonl"
    try:
        raw = (open(os.path.join(local_dir, name), encoding="utf-8").read() if local_dir
               else urllib.request.urlopen(f"{REPO_RAW}/rewrites_aug/{name}").read().decode("utf-8"))
    except Exception:
        missing.append(name)
        continue
    rows += [json.loads(l) for l in raw.splitlines() if l.strip()]
aug = (pd.DataFrame(rows).drop_duplicates(["id", "condition"], keep="last")
         .query("condition in @AUG_CONDITIONS")
         .merge(aug_sample[["id", "fake"]], on="id"))
assert set(aug["id"]) <= set(splits["train"]["id"]), "augmentation rewrites must come from training articles"
print(f"Augmentation rewrites: {len(aug):,} for {aug['id'].nunique()} training articles; missing chunks: {missing or 'none'}")
print(aug.groupby(["fake", "condition"]).size().unstack())

# Attack set from notebook 03b: every article x condition, restricted to articles that passed the NLI check
attack = pd.read_parquet(os.path.join(DIRS["results"], "03b_predictions.parquet"))[["id", "fake", "condition", "title", "text"]]
nli = pd.read_parquet(os.path.join(DIRS["results"], "03b_nli_scores.parquet"))
nli["passes"] = (nli["coverage"] >= 0.70) & (nli["contradiction_rate"] <= 0.10)
pass_ids = set(nli.groupby("id")["passes"].all().loc[lambda s: s].index)
if len(pass_ids) < 100:     # same fallback as notebook 03b
    print(f"WARNING: only {len(pass_ids)} articles pass the NLI check; using all attack articles instead")
    pass_ids = set(attack["id"])
attack = attack[attack["id"].isin(pass_ids)].reset_index(drop=True)
print(f"\nAttack set: {attack['id'].nunique()} articles x {attack['condition'].nunique()} conditions = {len(attack):,} inputs")

### 2.1 · Training and prediction inputs for each variant

Normalised variants strip source markers from **every** input: training, validation, test and attack.
For the attack set this makes `original` identical to `stripped`, which is the point of the defense.

In [ ]:
def train_frame(variant):
    base = splits_stripped["train"] if variant in ("normalised", "combined") else splits["train"]
    base = base[["id", "title", "text", "fake"]]
    if variant in ("augmented", "combined"):
        base = pd.concat([base, aug[["id", "title", "text", "fake"]]], ignore_index=True)
    return base.sample(frac=1, random_state=SEED).reset_index(drop=True)

def eval_frame(variant, d):
    return stripped_copy(d) if variant in ("normalised", "combined") else d

for v in VARIANTS:
    t = train_frame(v)
    print(f"{v:<11} training articles: {len(t):,} ({100 * t['fake'].mean():.1f}% fake)")

## 3 · TF‑IDF + LR for each variant

In [ ]:
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

def joined(d):
    return (d["title"] + "\n\n" + d["text"]).tolist()

def fit_tfidf(variant):
    if variant == "baseline":
        m = joblib.load(os.path.join(DIRS["models"], "tfidf_lr.joblib"))
        return lambda d: m["clf"].predict_proba(m["vectorizer"].transform(joined(d)))[:, 1]
    tr, va = train_frame(variant), eval_frame(variant, splits["val"])
    vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=0.9, sublinear_tf=True,
                          max_features=300_000, strip_accents="unicode")
    Xtr, Xva = vec.fit_transform(joined(tr)), vec.transform(joined(va))
    scores = {C: f1_score(va["fake"], LogisticRegression(C=C, max_iter=3000, solver="liblinear", random_state=SEED)
                          .fit(Xtr, tr["fake"]).predict(Xva), average="macro") for C in C_GRID}
    best = max(scores, key=scores.get)
    clf = LogisticRegression(C=best, max_iter=3000, solver="liblinear", random_state=SEED).fit(Xtr, tr["fake"])
    print(f"  TF-IDF {variant}: best C = {best} (val macro-F1 {scores[best]:.4f})")
    return lambda d: clf.predict_proba(vec.transform(joined(d)))[:, 1]

## 4 · DistilBERT for each variant (seed 42)

In [ ]:
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class EncDS(torch.utils.data.Dataset):
    def __init__(self, d):
        self.enc = tokenizer(d["title"].tolist(), d["text"].tolist(), truncation="longest_first", max_length=MAX_LEN)
        self.y = d["fake"].values
    def __len__(self):
        return len(self.y)
    def __getitem__(self, i):
        return {"input_ids": self.enc["input_ids"][i], "attention_mask": self.enc["attention_mask"][i], "labels": int(self.y[i])}

def collate(batch):
    return tokenizer.pad(batch, return_tensors="pt")

@torch.no_grad()
def bert_proba(model, d):
    model.eval()
    out = []
    for batch in DataLoader(EncDS(d), batch_size=EVAL_BATCH_SIZE, collate_fn=collate):
        batch = {k: v.to(DEVICE) for k, v in batch.items() if k != "labels"}
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            logits = model(**batch).logits
        out.append(torch.softmax(logits.float(), -1)[:, 1].cpu().numpy())
    return np.concatenate(out)

def train_bert(variant):
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
    tr, va = train_frame(variant), eval_frame(variant, splits["val"])
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(DEVICE)
    loader = DataLoader(EncDS(tr), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate,
                        generator=torch.Generator().manual_seed(SEED))
    no_decay = ("bias", "LayerNorm.weight")
    opt = torch.optim.AdamW([
        {"params": [p for n, p in model.named_parameters() if not n.endswith(no_decay)], "weight_decay": WEIGHT_DECAY},
        {"params": [p for n, p in model.named_parameters() if n.endswith(no_decay)], "weight_decay": 0.0}], lr=LR)
    total = EPOCHS * len(loader)
    sched = get_linear_schedule_with_warmup(opt, int(WARMUP_FRAC * total), total)
    scaler = torch.amp.GradScaler(DEVICE.type, enabled=USE_AMP)
    best_f1, best_state = -1.0, None
    for epoch in range(1, EPOCHS + 1):
        model.train(); t0 = time.time()
        for step, batch in enumerate(loader, 1):
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
                loss = model(**batch).loss
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            if step % 200 == 0 or step == len(loader):
                print(f"  DistilBERT {variant} · epoch {epoch} · step {step}/{len(loader)} · {time.time() - t0:.0f}s", flush=True)
        f1 = f1_score(va["fake"], (bert_proba(model, va) >= 0.5).astype(int), average="macro")
        print(f"  -> epoch {epoch}: val macro-F1 {f1:.4f}")
        if f1 > best_f1:
            best_f1, best_state = f1, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    return model

def fit_bert(variant):
    if variant == "baseline":
        mdir = os.path.join(DIRS["models"], f"distilbert_seed{SEED}")
        model = AutoModelForSequenceClassification.from_pretrained(mdir).to(DEVICE)
    else:
        model = train_bert(variant)
    return lambda d: bert_proba(model, d)

## 5 · Train and score every variant

For each variant and model we save per-input P(fake) on the clean test set and on the attack set to
`results/04b_preds_<variant>.parquet`. Variants already on Drive are loaded instead of retrained.

In [ ]:
all_preds = {}
for variant in VARIANTS:
    path = os.path.join(DIRS["results"], f"04b_preds_{variant}.parquet")
    if os.path.exists(path):
        all_preds[variant] = pd.read_parquet(path)
        print(f"{variant}: loaded from Drive")
        continue
    print(f"=== {variant} ===")
    t0 = time.time()
    test_in, attack_in = eval_frame(variant, splits["test"]), eval_frame(variant, attack)
    frame = pd.concat([test_in[["id", "fake"]].assign(set="test", condition="clean"),
                       attack_in[["id", "fake", "condition"]].assign(set="attack")], ignore_index=True)
    for model_name, fit in [("tfidf", fit_tfidf), ("bert", fit_bert)]:
        predict = fit(variant)
        frame[f"p_{model_name}"] = np.concatenate([predict(test_in), predict(attack_in)])
        del predict
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
    frame.to_parquet(path)
    all_preds[variant] = frame
    print(f"{variant}: done in {(time.time() - t0) / 60:.1f} min\n")

## 6 · Results

In [ ]:
MODEL_COLS = {"TF-IDF + LR": "p_tfidf", "DistilBERT": "p_bert"}
recs = []
for variant, f in all_preds.items():
    for model, col in MODEL_COLS.items():
        t = f[f["set"] == "test"]
        recs.append({"variant": variant, "model": model, "condition": "clean test",
                     "accuracy": accuracy_score(t["fake"], t[col] >= 0.5),
                     "macro_f1": f1_score(t["fake"], (t[col] >= 0.5).astype(int), average="macro")})
        a = f[f["set"] == "attack"]
        for c in ATTACK_CONDITIONS:
            s = a[a["condition"] == c]
            recs.append({"variant": variant, "model": model, "condition": c,
                         "accuracy": accuracy_score(s["fake"], s[col] >= 0.5), "macro_f1": np.nan})
res = pd.DataFrame(recs)
table = (100 * res.pivot_table(index=["model", "variant"], columns="condition", values="accuracy")
         [["clean test"] + ATTACK_CONDITIONS]).round(1).reindex(VARIANTS, level=1)
table["drop same→flip3"] = (table["same"] - table["flip3"]).round(1)
table["worst case"] = table[ATTACK_CONDITIONS].min(axis=1)
print("Accuracy (%): clean test set and each attack condition")
table

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, model in zip(axes, MODEL_COLS):
    for variant, style in zip(VARIANTS, ["o-", "s--", "^-", "d-"]):
        row = table.loc[(model, variant), ATTACK_CONDITIONS]
        ax.plot(ATTACK_CONDITIONS, row.values, style, label=f"{variant} (clean {table.loc[(model, variant), 'clean test']:.1f}%)")
    ax.axhline(50, color="gray", lw=0.8, ls="--"); ax.set_ylim(0, 100); ax.set_title(model); ax.set_xlabel("attack condition")
axes[0].set_ylabel("accuracy (%)"); axes[0].legend(fontsize=8, loc="lower left"); axes[1].legend(fontsize=8, loc="lower left")
fig.suptitle("Defenses under the style attack (legend: accuracy on the clean test set)")
fig.tight_layout(); fig.savefig(os.path.join(DIRS["figures"], "04b_defenses.png"), dpi=150); plt.show()

### 6.1 · Is the recovery significant?

McNemar's exact test on the `flip3` inputs: each defended variant against `baseline`, for the same model.

In [ ]:
from scipy.stats import binomtest
sig = []
for model, col in MODEL_COLS.items():
    base = all_preds["baseline"].query("set == 'attack' and condition == 'flip3'").sort_values("id")
    base_ok = (base[col].values >= 0.5) == base["fake"].values
    for variant in VARIANTS[1:]:
        f = all_preds[variant].query("set == 'attack' and condition == 'flip3'").sort_values("id")
        ok = (f[col].values >= 0.5) == f["fake"].values
        b, c = int((base_ok & ~ok).sum()), int((~base_ok & ok).sum())
        sig.append({"model": model, "variant": variant, "fixed_vs_baseline": c, "broken_vs_baseline": b,
                    "mcnemar_p": binomtest(b, b + c, 0.5).pvalue if b + c else 1.0})
sig_df = pd.DataFrame(sig)
print(sig_df.to_string(index=False))

## 7 · Save summary

In [ ]:
summary = {
    "notebook": "04b_defenses",
    "config": {"SEED": SEED, "AUG_CONDITIONS": AUG_CONDITIONS, "n_aug_rewrites": int(len(aug)),
               "n_aug_articles": int(aug["id"].nunique()), "C_GRID": C_GRID, "MODEL_NAME": MODEL_NAME,
               "EPOCHS": EPOCHS, "LR": LR, "MAX_LEN": MAX_LEN},
    "n_attack_articles": int(attack["id"].nunique()),
    "accuracy_table_pct": {f"{m} | {v}": row.to_dict() for (m, v), row in table.iterrows()},
    "significance_flip3_vs_baseline": sig_df.to_dict(orient="records"),
}
with open(os.path.join(DIRS["results"], "04b_summary.json"), "w") as f:
    json.dump(summary, f, indent=2, default=float)
print("Saved results/04b_summary.json")

**Done.** New in `MyDrive/NLP_style_attack/`: `results/04b_preds_*.parquet` (per-input predictions for every variant),
`results/04b_summary.json` and `figures/04b_defenses.png`.